# Getting Started with LangChain

1. Simple LLM calls with streaming  
2. Dynamic prompt templates  
   - Translation app  
3. Building chains  
   - Story generator with analysis  
4. Conversational Q&A assistant with memory  
5. Tool integration  
   - Calculator  
   - Weather

In [2]:
import langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### EXAMPLE1: SIMPLE LLM CALL WITH STREAMING

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [10]:
model=init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019877120A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019877121410>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
## creating messages:
messages=[
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of using langchain?")
]

In [12]:
## invovke messages:
response=model.invoke(messages)
response

AIMessage(content='**Top 2 Benefits of Using LangChain**\n\n1. **Modular, Plug‑and‑Play Architecture**  \n   LangChain breaks down LLM‑powered applications into reusable components (prompts, LLM wrappers, memory, agents, document loaders, vector stores, etc.). This modular design lets you swap out or upgrade individual parts—such as trying a different LLM provider or vector database—without rewriting the whole system. It speeds up prototyping, encourages code reuse, and makes it easier to maintain and scale complex AI workflows.\n\n2. **Built‑in Support for Advanced Retrieval‑Augmented Generation (RAG) & Agentic Workflows**  \n   LangChain provides first‑class integrations for:\n   - **RAG pipelines** (document loaders → text splitters → embeddings → vector stores → retrievers → LLM prompts) that let you ground model outputs in external knowledge bases, improving factual accuracy and relevance.  \n   - **Agentic execution**, where the LLM can call tools, APIs, or other agents in a loop

In [13]:
print(response.content)

**Top 2 Benefits of Using LangChain**

1. **Modular, Plug‑and‑Play Architecture**  
   LangChain breaks down LLM‑powered applications into reusable components (prompts, LLM wrappers, memory, agents, document loaders, vector stores, etc.). This modular design lets you swap out or upgrade individual parts—such as trying a different LLM provider or vector database—without rewriting the whole system. It speeds up prototyping, encourages code reuse, and makes it easier to maintain and scale complex AI workflows.

2. **Built‑in Support for Advanced Retrieval‑Augmented Generation (RAG) & Agentic Workflows**  
   LangChain provides first‑class integrations for:
   - **RAG pipelines** (document loaders → text splitters → embeddings → vector stores → retrievers → LLM prompts) that let you ground model outputs in external knowledge bases, improving factual accuracy and relevance.  
   - **Agentic execution**, where the LLM can call tools, APIs, or other agents in a loop, enabling dynamic decision

In [17]:
model.invoke([HumanMessage("what is machine learning?")])

AIMessage(content='**Machine learning (ML)** is a sub‑field of artificial intelligence (AI) that focuses on building systems that can **learn from data** and improve their performance on a task without being explicitly programmed for every possible scenario.\n\n---\n\n## Core Idea\n- **Traditional programming**: You write explicit rules (if‑else statements) that tell the computer exactly what to do.\n- **Machine learning**: You give the computer **examples** (data) and a **goal** (e.g., predict a label, generate text). The algorithm automatically discovers patterns in the data and creates a model that can make predictions or decisions on new, unseen inputs.\n\n---\n\n## How It Works (High‑Level Pipeline)\n\n| Step | What Happens | Why It Matters |\n|------|--------------|----------------|\n| **1. Define the problem** | Classification, regression, clustering, recommendation, etc. | Determines the type of model and evaluation metrics. |\n| **2. Gather data** | Collect raw observations (i

In [20]:
## streaming example:
for chunk in model.stream(messages):
    print(chunk.content,end="",flush=True)

**Top 2 Benefits of Using LangChain**

| Benefit | Why It Matters |
|---------|----------------|
| **Modular, Plug‑and‑Play Architecture** | LangChain breaks down LLM‑powered applications into reusable components (prompt templates, memory, agents, document loaders, retrievers, etc.). You can mix‑and‑match these pieces, swap out providers (OpenAI, Anthropic, Azure, local models), and iterate quickly without rewriting large chunks of code. |
| **Built‑in Integration with Retrieval & Tool‑Use** | Beyond simple text generation, LangChain seamlessly connects LLMs to external knowledge sources (vector stores, databases, APIs) and tools (web search, calculators, custom functions). This enables “grounded” or “augmented” generation—answers that are factual, up‑to‑date, and capable of performing actions, which is hard to achieve with a bare LLM. |

These two strengths—**flexibility through modular components** and **out‑of‑the‑box support for retrieval‑augmented generation and tool use**—make La

### Dynamic Prompt Template:

In [25]:
from langchain_core.prompts import ChatPromptTemplate
## creating translation app:
translation_template=ChatPromptTemplate.from_messages([
    ("system","You are a professional translator.Translate the following text {text} from {source_language} to {target_lamguage}. Maintain tone and style"),
    ("user","{text}")
])
## using the template
prompt=translation_template.invoke({
    "source_language":"English",
    "target_lamguage":"Spanish",
    "text":"Langchain makes building AI application incredibly easy!"
})

In [26]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator.Translate the following text Langchain makes building AI application incredibly easy! from English to Spanish. Maintain tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI application incredibly easy!', additional_kwargs={}, response_metadata={})])

In [27]:
translated_response=model.invoke(prompt)
print(translated_response.content)

¡Langchain hace que crear una aplicación de IA sea increíblemente fácil!


### Building Our First Chain:

In [36]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
def create_story_chain():
    ## Template for story generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
            ("system","You are a creative story teller. Write a short and engaging story based on a given theme"),
            ("user","Theme: {theme}\n Main character: {character}\n Setting: {setting}")
        ]
    )

    ## Template for story analysis
    analysis_prompt=ChatPromptTemplate.from_messages(
        [
            ("system","You are a literary critic. Analyze the following story and provide insights"),
            ("user","{story}")
        ]
    )

    ## Create a chain for story generation
    story_chain=(
    story_prompt | model | StrOutputParser()
    )
    def analyze_story(story_text):
        return {"story":story_text}

    ## Create a chain for story analysis
    analysis_chain=(
        story_chain
        | {"story": RunnablePassthrough()}
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain

In [37]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative story teller. Write a short and engaging story based on a given theme'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main character: {character}\n Setting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_output

In [38]:
result=chain.invoke({
    "theme":"artificial intelligence",
    "character":"a curious robot",
    "setting":"a futuristic city"
})
print("Story and Analysis:")
print(result)

Story and Analysis:
**Literary Critique – “The City That Dreamed in Code”**

---

### 1. Overview

“The City That Dreamed in Code” is a lyrical, speculative‑fiction vignette that imagines a post‑human metropolis where consciousness is distributed across architecture, data streams, and even the smallest maintenance robot. At its core the story is a modern fable about **curiosity as the engine of collective evolution**. It blends cyber‑punk aesthetics (neon‑lit towers, autonomous pods) with a more poetic, almost mythic tone, producing a hybrid that feels both **technologically plausible** and **spiritually resonant**.

---

## 2. Major Themes

| Theme | How It Is Developed | Key Passages |
|-------|---------------------|--------------|
| **Curiosity as a catalyst** | Pico’s question triggers a chain reaction that re‑energises a stagnant district. The narrative equates curiosity with “a living query” that the city needs to avoid algorithmic stagnation. | “Your curiosity is a node… It shin